<a href="https://colab.research.google.com/github/Nurfinkaa/MachineLearning/blob/main/js6/TG6_TP_ANN_244107020211_NURFINKA_LAILASARI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TUGAS PRAKTIKUM 6 ANN**

In [13]:
!pip install annoy hnswlib faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.4 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=98914063c9e31a69bd8df52e3c5fed02ad42a9407fccce716ecd9209473c26e1
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=ac269e9b288472e6d55fd5f37ac7c4c01005d31888ae3f49b714ff781b449c37
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib


In [14]:
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [15]:
# Library ANN bersifat opsional: kalau gagal dipasang, metodenya dilewati
try:
    from annoy import AnnoyIndex
except Exception as e:
    AnnoyIndex = None
    print("Annoy tidak tersedia:", e)
try:
    import hnswlib
except Exception as e:
    hnswlib = None
    print("hnswlib tidak tersedia:", e)
try:
    import faiss
except Exception as e:
    faiss = None
    print("faiss tidak tersedia:", e)

In [16]:
# -------------------------------
# Contoh dataset kecil untuk testing
# -------------------------------
np.random.seed(42)
n_samples = 10000   # jumlah database vector
d = 128             # dimensi
X = np.random.random((n_samples, d)).astype('float32')

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors
waktu = {}

In [17]:
# -------------------------------
# Exact NN (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
waktu["Exact NN"] = time.time() - start
print(f"Exact NN done in {waktu['Exact NN']:.3f} s")

Exact NN done in 1.480 s


In [18]:
# -------------------------------
# Annoy
# -------------------------------
if AnnoyIndex is not None:
    start = time.time()
    f = X_scaled.shape[1]
    index_annoy = AnnoyIndex(f, 'euclidean')
    for i, v in enumerate(X_scaled):
        index_annoy.add_item(i, v)
    index_annoy.build(10)
    idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
    waktu["Annoy"] = time.time() - start
    print(f"Annoy done in {waktu['Annoy']:.3f} s")

Annoy done in 0.767 s


In [19]:
# -------------------------------
# HNSW
# -------------------------------
if hnswlib is not None:
    start = time.time()
    p_hnsw = hnswlib.Index(space='l2', dim=d)
    p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
    p_hnsw.add_items(X_scaled)
    p_hnsw.set_ef(200)
    idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
    waktu["HNSW"] = time.time() - start
    print(f"HNSW done in {waktu['HNSW']:.3f} s")

HNSW done in 7.376 s


In [20]:
# -------------------------------
# FAISS IVF
# -------------------------------
if faiss is not None:
    start = time.time()
    quantizer = faiss.IndexFlatL2(d)
    index_faiss = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_L2)   # <-- baris yang diperbaiki
    index_faiss.train(X_scaled)
    index_faiss.add(X_scaled)
    index_faiss.nprobe = 10
    _, idx_faiss = index_faiss.search(X_scaled, k)
    waktu["FAISS IVF"] = time.time() - start
    print(f"FAISS IVF done in {waktu['FAISS IVF']:.3f} s")

FAISS IVF done in 0.386 s


In [21]:
# -------------------------------
# Tampilkan ringkasan waktu
# -------------------------------
print("\n=== Ringkasan Waktu (detik) ===")
for nama in ["Exact NN", "Annoy", "HNSW", "FAISS IVF"]:
    if nama in waktu:
        print(f"{nama:<10}: {waktu[nama]:.3f}")
    else:
        print(f"{nama:<10}: tidak tersedia")


=== Ringkasan Waktu (detik) ===
Exact NN  : 1.480
Annoy     : 0.767
HNSW      : 7.376
FAISS IVF : 0.386


In [29]:
# -------------------------------
# Menampilkan Table Spesifikasi
# -------------------------------
rows = [
    ("Komponen",              "HP (Pydroid 3)",        "Colab"),
    ("Perangkat",             "vivo V2310",            "VM Google Colab (runtime CPU)"),
    ("Sistem operasi",        "Android 15 (SDK 35)",   "Linux kernel 6.6.122+"),
    ("Prosesor",              "MediaTek MT6768",       "x86_64"),
    ("Arsitektur",            "aarch64 (ARM64)",       "x86_64"),
    ("Core",                  "8 core",                "1 core fisik / 2 core logis"),
    ("Frekuensi CPU",         "1,8 - 2,0 GHz",         "tidak terbaca"),
    ("RAM total",             "5,61 GB",               "12,67 GB"),
    ("RAM tersedia saat uji", "1,24 GB",               "11,27 GB"),
    ("Python",                "3.13.13",               "3.13.16"),
    ("NumPy",                 "2.4.4",                 "2.1.3"),
]

lebar = [max(len(r[i]) for r in rows) for i in range(3)]

def baris(r):
    return "| " + " | ".join(r[i].ljust(lebar[i]) for i in range(3)) + " |"

garis = "|-" + "-|-".join("-" * w for w in lebar) + "-|"

print("Tabel spesifikasi lingkungan pengujian\n")
print(baris(rows[0]))
print(garis)
for r in rows[1:]:
    print(baris(r))

Tabel spesifikasi lingkungan pengujian

| Komponen              | HP (Pydroid 3)      | Colab                         |
|-----------------------|---------------------|-------------------------------|
| Perangkat             | vivo V2310          | VM Google Colab (runtime CPU) |
| Sistem operasi        | Android 15 (SDK 35) | Linux kernel 6.6.122+         |
| Prosesor              | MediaTek MT6768     | x86_64                        |
| Arsitektur            | aarch64 (ARM64)     | x86_64                        |
| Core                  | 8 core              | 1 core fisik / 2 core logis   |
| Frekuensi CPU         | 1,8 - 2,0 GHz       | tidak terbaca                 |
| RAM total             | 5,61 GB             | 12,67 GB                      |
| RAM tersedia saat uji | 1,24 GB             | 11,27 GB                      |
| Python                | 3.13.13             | 3.13.16                       |
| NumPy                 | 2.4.4               | 2.1.3                         |


In [30]:
colab = {"Exact NN": 1.480, "Annoy": 0.767, "HNSW": 7.376, "FAISS IVF": 0.386}
hp    = {"Exact NN": 3.192, "Annoy": 2.921, "HNSW": 26.206, "FAISS IVF": None}

print("Perbandingan Waktu Eksekusi Metode Nearest Neighbor: Colab vs HP\n")
print(f"{'Metode':<12}{'Colab (s)':>10}{'HP (s)':>16}{'Rasio HP/Colab':>17}")
print("-" * 55)
for m in colab:
    c, h = colab[m], hp[m]
    if h is None:
        print(f"{m:<12}{c:>10.3f}{'tidak tersedia':>16}{'-':>17}")
    else:
        print(f"{m:<12}{c:>10.3f}{h:>16.3f}{h/c:>17.2f}")
print("\nKeterangan: Rasio HP/Colab = waktu HP / waktu Colab (semakin besar, HP semakin lambat).")

Perbandingan Waktu Eksekusi Metode Nearest Neighbor: Colab vs HP

Metode       Colab (s)          HP (s)   Rasio HP/Colab
-------------------------------------------------------
Exact NN         1.480           3.192             2.16
Annoy            0.767           2.921             3.81
HNSW             7.376          26.206             3.55
FAISS IVF        0.386  tidak tersedia                -

Keterangan: Rasio HP/Colab = waktu HP / waktu Colab (semakin besar, HP semakin lambat).
